# EDA Checklist Before Modelling

In [41]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\maxan\OneDrive\Desktop\0. Personal Projects\market-intelligence-pipeline")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

## 1. Create EDA notebook
- [ ] Create notebooks/01_eda.ipynb
- [ ] Load price_features from DuckDB
- [ ] Sort by ticker, date
- [ ] Confirm row count and column count
- [ ] Confirm date range
- [ ] Confirm tickers included

In [42]:
import pandas as pd
import numpy as np
import duckdb

from src.config import (
    DATABASE_PATH,
    ROLLING_WINDOWS,
    LAGGED_WINDOWS,
    MA_WINDOWS,
    TICKERS,
    START_DATE,
    END_DATE
)

from src.features import (
    build_price_feature_columns
)

CALENDAR = "no_calendar"


def pull_calendar_table(table_name: str, calendar: str) -> pd.DataFrame:

    print("[START] Connecting to database...")

    con = duckdb.connect(DATABASE_PATH)

    print(f"[START] Reading {table_name}")

    df = con.sql(f"""
    
        SELECT *
        FROM {table_name}

    """).df()

    print(f"[DONE] Read {table_name}")

    con.close()

    print(f"[START] Building {CALENDAR} required columns...")

    base_cols = [
        "ticker",
        "date",
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume"
    ]

    feature_columns = build_price_feature_columns(
        calendars = [CALENDAR],
        rolling_windows = ROLLING_WINDOWS,
        lagged_windows = LAGGED_WINDOWS,
        ma_windows = MA_WINDOWS
    )

    expected_columns = base_cols + list(feature_columns.keys())

    calendar_df = df[expected_columns].copy()

    print(f"[DONE] Filtered to {CALENDAR}")

    return calendar_df

In [43]:
# Load price_features from DuckDB

df = pull_calendar_table(
    table_name = "price_features", 
    calendar = CALENDAR
    )

print(f"[STEP] Loaded price_features table selecting {CALENDAR} columns")
print(df.head(10))

[START] Connecting to database...
[START] Reading price_features
[DONE] Read price_features
[START] Building no_calendar required columns...
[DONE] Filtered to no_calendar
[STEP] Loaded price_features table selecting no_calendar columns
  ticker       date        open        high         low       close  \
0    GLD 2018-01-04  124.889999  125.849998  124.739998  125.459999   
1    GLD 2018-01-19  126.570000  126.730003  126.410004  126.419998   
2    GLD 2018-01-25  128.690002  129.509995  127.360001  127.970001   
3    GLD 2018-02-12  125.190002  125.820000  125.110001  125.370003   
4    GLD 2018-02-14  126.470001  128.589996  126.290001  128.229996   
5    GLD 2018-02-20  127.279999  127.400002  126.040001  126.239998   
6    GLD 2018-02-26  126.449997  126.620003  126.180000  126.449997   
7    GLD 2018-03-08  125.690002  125.699997  125.129997  125.419998   
8    GLD 2018-05-01  123.900002  123.980003  123.389999  123.709999   
9    GLD 2018-05-09  124.449997  124.870003  124.2399

In [44]:
# Sort by ticker, date.

df = df.sort_values(["ticker", "date"]).reset_index(drop = True)

print("[STEP] Sorted by ticker, date")
print(df.head(10))

[STEP] Sorted by ticker, date
  ticker       date        open        high         low       close  \
0    GLD 2018-01-02  124.660004  125.180000  124.389999  125.150002   
1    GLD 2018-01-03  125.050003  125.089996  124.099998  124.820000   
2    GLD 2018-01-04  124.889999  125.849998  124.739998  125.459999   
3    GLD 2018-01-05  124.930000  125.480003  124.830002  125.330002   
4    GLD 2018-01-08  125.199997  125.320000  124.900002  125.309998   
5    GLD 2018-01-09  124.489998  124.860001  124.230003  124.730003   
6    GLD 2018-01-10  125.169998  125.309998  124.720001  125.029999   
7    GLD 2018-01-11  125.370003  125.660004  125.250000  125.440002   
8    GLD 2018-01-12  126.010002  127.129997  125.809998  126.959999   
9    GLD 2018-01-16  126.599998  127.180000  126.400002  127.169998   

    adj_close    volume  daily_return_no_calendar  log_return_no_calendar  \
0  125.150002  11762500                       NaN                     NaN   
1  124.820000   7904300           

In [45]:
# Confirm row count and column count

shape = df.shape

print(f"Row count: {shape[0]}")
print(f"Column count: {shape[1]}")

Row count: 11323
Column count: 23


In [46]:
# Confirm date range

max_date = df["date"].dt.date.max()
min_date = df["date"].dt.date.min()

print(f"Date range: {min_date} -> {max_date}")

print(f"CONFIG Date range: {START_DATE} -> {END_DATE}")

print(f"""
Yfinance pulls dates one day after CONFIG START_DATE
and CONFIG END_DATE when set to None pulls data till today
""")

Date range: 2018-01-02 -> 2026-05-15
CONFIG Date range: 2018-01-01 -> None

Yfinance pulls dates one day after CONFIG START_DATE
and CONFIG END_DATE when set to None pulls data till today



In [47]:
# Confirm tickers included

tickers = df["ticker"].unique().tolist()

print(f"Tickers included: {sorted(tickers)}")

print(f"\nCONFIG Tickers: {sorted(TICKERS)}")

Tickers included: ['GLD', 'MU', 'NKE', 'RPI.L', 'SNDK', 'SPY', 'TLT']

CONFIG Tickers: ['GLD', 'MU', 'NKE', 'RPI.L', 'SNDK', 'SPY', 'TLT']


## 2. Data quality checks
- [ ] Check missing values by column
- [ ] Check missing values by ticker
- [ ] Check duplicate ticker + date rows
- [ ] Check infinite values
- [ ] Check zero/negative prices
- [ ] Check date gaps per ticker
- [ ] Check whether all tickers have enough history
- [ ] Check target columns have expected nulls at the final row per ticker

In [48]:
# Missing values by column

print("Number of missing values per column:")

df.isna().sum()

Number of missing values per column:


ticker                                  0
date                                    0
open                                    0
high                                    0
low                                     0
close                                   0
adj_close                               0
volume                                  0
daily_return_no_calendar                7
log_return_no_calendar                  7
cumulative_returns_no_calendar          7
rolling_7d_return_no_calendar          49
rolling_30d_return_no_calendar        210
lag_1_return_no_calendar               14
lag_5_return_no_calendar               42
moving_avg_20_no_calendar             133
moving_avg_50_no_calendar             343
price_vs_ma20_no_calendar             133
relative_volume_no_calendar           133
rolling_30d_volatility_no_calendar    210
drawdown_no_calendar                    0
target_next_day_return_no_calendar      7
target_direction_no_calendar            7
dtype: int64

In [53]:
# Missing values by ticker

print("Missing values by ticker:")

df.groupby("ticker").apply(lambda g: g.isna().sum().sum())

Missing values by ticker:


ticker
GLD      186
MU       186
NKE      186
RPI.L    186
SNDK     186
SPY      186
TLT      186
dtype: int64

In [50]:
# Check duplicate ticker + date rows

print("Number of duplicate ticker, date rows: ", df.duplicated(subset=["ticker", "date"]).sum())

Number of duplicate ticker, date rows:  0


In [63]:
# Check infinite values

numeric_df = df.select_dtypes(include="number")

total_inf = np.isinf(numeric_df).sum()

print("Number of infinite values in numerical columns:")

print(total_inf)

Number of infinite values in numerical columns:
open                                  0
high                                  0
low                                   0
close                                 0
adj_close                             0
volume                                0
daily_return_no_calendar              0
log_return_no_calendar                0
cumulative_returns_no_calendar        0
rolling_7d_return_no_calendar         0
rolling_30d_return_no_calendar        0
lag_1_return_no_calendar              0
lag_5_return_no_calendar              0
moving_avg_20_no_calendar             0
moving_avg_50_no_calendar             0
price_vs_ma20_no_calendar             0
relative_volume_no_calendar           0
rolling_30d_volatility_no_calendar    0
drawdown_no_calendar                  0
target_next_day_return_no_calendar    0
target_direction_no_calendar          0
dtype: Int64


In [69]:
# Check zero / negative prices:

price_df = df[["open", "high", "low", "close", "adj_close"]]

zero_negative_count = (price_df <= 0).sum().sum()

print(f"Number of zero or negative prices: {zero_negative_count}")

Number of zero or negative prices: 0


In [73]:
# Check date gaps per ticker:

group_col = "ticker"
date_col = "date"

date_gap_df = df.copy()

## Convert date to date time format just in case is text.
date_gap_df[date_col] = pd.to_datetime(date_gap_df[date_col])

## Sort by ticker then date just in case.
date_gap_df = date_gap_df.sort_values([group_col, date_col])

## Get previous date from wihtin each group by shifting date down.
date_gap_df["previous_date"] = date_gap_df.groupby(group_col)[date_col].shift()

## Calculate date gap between sequential rows.
date_gap_df["gap"] = date_gap_df[date_col] - date_gap_df["previous_date"]

## Identify any gap bigger than a 1 day gap.
gaps = date_gap_df[date_gap_df["gap"] > pd.Timedelta(days=1)]

## Count gaps per ticker
gap_count = gaps.groupby(group_col).size()

print("Number of date gaps per ticker:")
print(gap_count)

Number of date gaps per ticker:
ticker
GLD      460
MU       460
NKE      460
RPI.L    103
SNDK      69
SPY      460
TLT      460
dtype: int64


In [ ]:
# Check sufficent data per ticker?

df_history = df.copy()

date_col = "date"
group_col = "ticker"

## Ensure date time format
df_history[date_col] = pd.to_datetime(df_history[date_col])

## Count rows per ticker
df_history.groupby(group_col).size()

## Create summary dataframe per ticker
history_summary = (
    df_history.groupby(group_col)
    .agg(
        first_date = ("date", "min"),
        last_date = ("date", "max"),
        observations = ("date", "size")
    )
)

## Create list of rolling windows
windows = ROLLING_WINDOWS + LAGGED_WINDOWS + MA_WINDOWS

## Define sufficeint date as 2*max window
history_summary["sufficient_data"] = history_summary["observations"] > (2 * max(windows))

history_summary = history_summary.sort_values("observations")

history_summary


,first_date,last_date,observations,sufficient_data
ticker,,,,
SNDK,2025-02-13,2026-05-15,315,True
RPI.L,2024-06-11,2026-05-15,488,True
MU,2018-01-02,2026-05-15,2104,True
GLD,2018-01-02,2026-05-15,2104,True
NKE,2018-01-02,2026-05-15,2104,True
SPY,2018-01-02,2026-05-15,2104,True
TLT,2018-01-02,2026-05-15,2104,True


In [ ]:
# Check target columns have expected nulls at final row per ticker

## Select columsn that contain ticker or target
target_df = df.loc[:, df.columns.str.contains("target|ticker", case=False)]

## Select last row per group to inspect
target_df.groupby("ticker").tail(1)

,ticker,target_next_day_return_no_calendar,target_direction_no_calendar
2103,GLD,NaN,<NA>
4207,MU,NaN,<NA>
6311,NKE,NaN,<NA>
6799,RPI.L,NaN,<NA>
7114,SNDK,NaN,<NA>
9218,SPY,NaN,<NA>
11322,TLT,NaN,<NA>
